In [70]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import get_scorer_names
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier

In [44]:
df_train=pd.read_csv('data/train.csv')
df_test=pd.read_csv('data/test.csv')

In [45]:
le=LabelEncoder()

X=df_train.drop(columns=['health_condition'])
y = le.fit_transform(df_train['health_condition']) 
test_ids = df_test['id'].copy()

In [46]:
def features(df):
    df=df.copy()
    df.drop(columns='id')
    df['stress_level_NaN']=df['stress_level'].isna().astype(int)
    df['sleep_duration_NaN']=df['sleep_duration'].isna().astype(int)
    df['physical_activity_level_NaN']=df['physical_activity_level'].isna().astype(int)
    df['smoking_alcohol_NaN']=df['smoking_alcohol'].isna().astype(int)
    df['calorie_expenditure_NaN']=df['calorie_expenditure'].isna().astype(int)
    df['water_intake_NaN']=df['water_intake'].isna().astype(int)
    return df

features_eng=FunctionTransformer(features)

In [47]:
ord_cols=['sleep_quality','stress_level','smoking_alcohol','physical_activity_level']

obj_cols=X.select_dtypes(include=['object']).columns
cat_cols=[c for c in obj_cols if c not in ord_cols]

bool_cols=X.select_dtypes(include=['bool']).columns
num_cols=X.select_dtypes(include=['number']).columns

In [49]:
ordi_scales={
    'sleep_quality':['poor','average','good'],
    'stress_level':['high','medium','low'],
    'smoking_alcohol':['yes','occasional','no'],
    'physical_activity_level':['sedentary', 'moderate', 'active']
}

In [50]:
ordinal_pipe=Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),
('OrdinalEncoder',OrdinalEncoder(categories=[ordi_scales[c] for c in ord_cols]))
])

cat_pipe=Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),
('OneHotEncoder',OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

num_pipe=Pipeline([('imputer',SimpleImputer(strategy='median'))
])

In [71]:
preprocessor=ColumnTransformer([('ordinal_pipe',ordinal_pipe,ord_cols),
('cat_pipe',cat_pipe,cat_cols),
('num_pipe',num_pipe,num_cols)])

logistic_pipe=Pipeline([('features_eng',features_eng),
('preprocessor',preprocessor),
('scaler',StandardScaler()),
('logistic',LogisticRegression(random_state=42,max_iter=1000,class_weight='balanced'))])

In [73]:
scores=cross_val_score(logistic_pipe,X,y,cv=5,scoring='balanced_accuracy')
print("LogisticRegression :", scores.mean(), "+/-", scores.std())

LogisticRegression : 0.8569847536243339 +/- 0.0010331655354212383


In [74]:
random_pipe=Pipeline([('features_eng',features_eng),
('preprocessor',preprocessor),
('scaler',StandardScaler()),
('randomforest',RandomForestClassifier(random_state=42,class_weight='balanced',n_jobs=-1))])

In [ ]:
scores=cross_val_score(random_pipe,X,y,cv=5,scoring='balanced_accuracy')
print("RandomForest :", scores.mean(), "+/-", scores.std())

In [ ]:
light_pipe=Pipeline([('features_eng',features_eng),
('preprocessor',preprocessor),
('scaler',StandardScaler()),
('lgbm',LGBMClassifier(random_state=42,class_weight='balanced'))])


In [ ]:
scores=cross_val_score(random_pipe,X,y,cv=5,scoring='balanced_accuracy')
print("LightGBM :", scores.mean(), "+/-", scores.std())